# MRA_TIP on Kaggle: train and evaluate one configuration

**Before running**
1. *Add Input* → attach the **scenes** dataset (output of the step-2 subset notebook; it contains `kitti_subset/selection.json`).
2. Optional: attach an earlier run's dataset holding its `results_trained.json` for a later comparison.
3. *Settings*: Accelerator **GPU T4**, Internet **On**.
4. Set `RUN` below (`mra_before`) and click *Run All*. When `Epoch [1][50/400]` lines appear, stop the session and use *Save Version → Save & Run All (Commit)* so the ~5 h run continues in the background.

All runs start from MRA's own initial weights (ImageNet ResNet-50, rest default init) and train with `--deterministic`, as `tools/dist_train.sh` does.

Everything this notebook does is in the repository's `kaggle/` folder; see `kaggle/README.md`.

In [ ]:
# ---- Settings ----
RUN      = "mra_before"     # which kaggle/configs/<RUN>.py to train
REPO_URL = "https://github.com/<your-account>/MRA_TIP.git"
BRANCH   = "main"
EXTRA    = ""                 # extra train.py args, e.g. "--cfg-options runner.max_epochs=3"
import os
REPO = f"/kaggle/working/MRA_TIP"
os.environ.update(dict(RUN=RUN, REPO=REPO, REPO_URL=REPO_URL, BRANCH=BRANCH, EXTRA=EXTRA))
print("run:", RUN, "| repo:", REPO)

In [ ]:
%%bash
# 1. Code + software (~15 min the first time; skipped when already installed in this session)
nvidia-smi --query-gpu=name,memory.total --format=csv
[ -d "$REPO" ] || git clone -q -b "$BRANCH" "$REPO_URL" "$REPO"
cd "$REPO" && git log --oneline -1
bash kaggle/install.sh && bash kaggle/get_weights.sh

In [ ]:
# 2. Data: find the saved subset and rebuild MRA's file layout at /tmp/kitti_full/ (~3 min)
import glob, json, subprocess, os
hits = glob.glob("/kaggle/input/**/kitti_subset/selection.json", recursive=True)
assert hits, "scenes dataset not attached (see the top of this notebook)"
SRC = os.path.dirname(hits[0]) + "/"
s = json.load(open(hits[0]))["summary"]
print("subset:", SRC, f"| train frames {s['train']['frames']} | val frames {s['val']['frames']}")
if not os.path.exists("/tmp/kitti_full/dataset/labels"):
    subprocess.run(["python", f"{REPO}/kaggle/expand_subset.py", SRC, "/tmp/kitti_full/"], check=True)

In [ ]:
# 3. Initial weights are MRA's own: nothing to prepare. An attached earlier results_trained.json is only located here
#    (compare with RaCA/kaggle/compare.py).
import glob, os
BEFORE = glob.glob("/kaggle/input/**/results_trained.json", recursive=True)
os.environ["BEFORE_JSON"] = BEFORE[0] if BEFORE else ""
print("'before' results for comparison:", BEFORE[0] if BEFORE else "none attached")


In [ ]:
%%bash
# 4. Train (~40-55 min per epoch on a T4). Lines like `Epoch [1][50/400] ... loss: ...` show progress.
cd "$REPO" && bash kaggle/train.sh kaggle/configs/$RUN.py $RUN $EXTRA

In [ ]:
%%bash
# 5. Evaluate the trained model on the 150 validation frames (~5 min)
cd "$REPO" && bash kaggle/eval.sh kaggle/configs/$RUN.py /kaggle/working/runs/$RUN/latest.pth /kaggle/working/runs/$RUN/results_trained.json "$RUN: trained" 

In [ ]:
%%bash
# 6. What is kept as the notebook output (checkpoint, logs, results); the dataset copy in /tmp is not saved
du -sh /kaggle/working/runs/$RUN; ls /kaggle/working/runs/$RUN